In [3]:
# Step 1: Tokenization
def tokenize(sentences):
    tokens = []
    for sentence in sentences:
        cleaned_sentence = re.sub(r'[^\w\s]', '', sentence).lower()  # Clean and lowercase the sentence
        tokens.extend(cleaned_sentence.split())  # Split into words
        filtered_tokens = [token for token in tokens if token not in stopwords]
    return filtered_tokens

# Step 2: Create Vocabulary
def create_vocabulary(tokens):
    return list(set(tokens))  # Unique list of words

# Step 3: Initialize Vectors (Equidistant placement for root words)
def initialize_vectors(vocabulary, root_words, dimensions=10, distance=2):
    vectors = {}
    num_root_words = len(root_words)
    angle_increment = 2 * math.pi / num_root_words  # Calculate angle increment for equidistant positioning

    # Step 3.1: Initialize root word vectors with equidistance
    for index, word in enumerate(root_words):
        angle = index * angle_increment
        vectors[word] = [distance * math.cos(angle), distance * math.sin(angle)] + [random.uniform(-1, 1) for _ in range(dimensions - 2)]

    # Step 3.2: Initialize other words with random vectors
    for word in vocabulary:
        if word not in vectors:
            vectors[word] = [random.uniform(-1, 1) for _ in range(dimensions)]

    return vectors

# Step 4: Create Clusters with Context Map
# Step 4: Create Clusters with Context Map and Update Context Word Vectors
def create_clusters(vocabulary, context_map, synonym_map, vectors, root_words):
    clusters = {}

    # Iterate over each root word to create clusters
    for root_word in root_words:
        clusters[root_word] = {
            "position": vectors[root_word],
            "subclusters": {}
        }

        # Create subclusters based on context words
        if root_word in context_map:
            for context, related_words in context_map[root_word].items():
                # Only use related words that are also in the vocabulary
                valid_related_words = [word for word in related_words if word in vocabulary]
                clusters[root_word]["subclusters"][context] = {
                    "words": valid_related_words,
                    "synonyms": synonym_map[root_word].get(context, [])
                }

                # Step 4.1: Update vectors for the context words
                for word in valid_related_words:
                    if word not in vectors:
                        # Initialize vector for the context word if not already present
                        vectors[word] = [random.uniform(-1, 1) for _ in range(len(vectors[root_word]))]
                    else:
                        # Optionally, update the vector for the context word (simple averaging for example)
                        vectors[word] = [(vectors[word][i] + vectors[root_word][i]) / 2 for i in range(len(vectors[word]))]

    return clusters


# List of common stopwords to remove
stopwords = set([
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", 
    "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", 
    "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", 
    "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", 
    "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", 
    "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", 
    "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", 
    "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", 
    "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", 
    "now"
])

# Helper function: Cosine similarity between two vectors
def cosine_similarity(vec1, vec2):
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    norm_vec1 = math.sqrt(sum(a * a for a in vec1))
    norm_vec2 = math.sqrt(sum(b * b for b in vec2))
    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0
    return dot_product / (norm_vec1 * norm_vec2)

import re
import random
import math

# List of common stopwords to remove
stopwords = set([
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", 
    "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", 
    "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", 
    "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", 
    "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", 
    "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", 
    "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", 
    "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", 
    "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", 
    "now"
])

# Helper function: Cosine similarity between two vectors
def cosine_similarity(vec1, vec2):
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    norm_vec1 = math.sqrt(sum(a * a for a in vec1))
    norm_vec2 = math.sqrt(sum(b * b for b in vec2))
    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0
    return dot_product / (norm_vec1 * norm_vec2)

# Modified Step 5: Process Input Sentence and Search in Subclusters (Match at least one adjacent word)
def process_input_subcluster(input_sentence, clusters, vectors):
    input_tokens = re.sub(r'[^\w\s]', '', input_sentence).lower().split()  # Clean and tokenize input sentence
    
    # Step 1: Remove stopwords
    filtered_tokens = [token for token in input_tokens if token not in stopwords]

    root_word = None
    adjacent_words = []

    # Step 5.1: Identify the root word from the filtered input tokens
    for token in filtered_tokens:
        if token in clusters:
            root_word = token
            break

    if not root_word:
        print("No root word found in the sentence.")
        return {}

    print(f"Root word identified: {root_word}")

    # Step 5.2: Select 2 adjacent words from the filtered input sentence
    for token in filtered_tokens:
        if token != root_word:
            adjacent_words.append(token)
        if len(adjacent_words) == 2:  # Stop once we have two adjacent words
            break

    # If fewer than 2 adjacent words are found, include the root word as one of the adjacent words
    if len(adjacent_words) < 2:
        adjacent_words.append(root_word)

    if len(adjacent_words) < 2:
        print("Not enough adjacent words found.")
        return {}

    print(f"Adjacent words selected: {adjacent_words}")

    # Step 5.3: Search in subclusters for context matching at least one adjacent word
   
    subclusters = {}
    if root_word in clusters:
        for context, cluster_info in clusters[root_word]["subclusters"].items():
            related_words = cluster_info["words"]
    
            # Check if at least one adjacent word is in the subcluster's related words
            if any(word in related_words for word in adjacent_words):
                # Fetch synonyms from the synonym_map for the specific context
                synonyms = synonym_map[root_word].get(context, [])
    
                subclusters[context] = {
                    "related_words": related_words,
                    "synonyms": synonyms  # Correctly fetch synonyms
                }


    if not subclusters:
        print("No subclusters found matching adjacent words.")
    return subclusters

# Example sentences and clusters can be added here



# Example sentences and clusters can be added her

# Example sentences and context map
sentences = [
    "The sun is bright.", "He is a bright student.",
    "The winter air is cold.", "His attitude was cold.",
    "The lamp is light.", "The box is light.",
    "The exam was hard.", "The rock is hard.",
    "He is a sharp student.", "The knife is sharp.",
    "The fire is warm.", "His welcome was warm.",
    "The load is heavy.", "The workload was heavy.",
    
    "The pillow is soft.", "Her voice was soft.",
    "The dessert is sweet.", "Her gesture was sweet.",
    "The breeze is cool.", "His style was cool.",
    "The car is fast.", "His response was fast.",
    "The table is flat.", "The lecture was flat.",
    
    "The ocean is deep.", "His thoughts were deep.",
    "The glass is clear.", "The instructions were clear.",
    "The music was loud.", "His statement was loud.",
    "The room is dark.", "The atmosphere was dark.",
    "The room is clean.", "His record was clean.",
    "The building is high.", "The temperature was high.",
    "The businessman is rich.", "The sauce was rich.",
    "The child is weak.", "The argument was weak.",
    "The desert is dry.", "His response was dry."
]

# Define the context map
context_map = {
    "bright": {
        "intelligent": ["student", "learner", "scholar", "person"],
        "shining": ["sun", "light", "lamp", "sky"]
    },
    "cold": {
        "unfriendly": ["attitude", "personality", "behavior", "demeanor"],
        "temperature": ["winter", "air", "ice"]
    },
    "light": {
        "illumination": ["room", "window", "lamp", "source"],
        "weight": ["feather", "object","box", "package", "load"]
    },
    "hard": {
        "difficult": ["exam", "task", "rock", "surface"],
        "solid": ["rock", "material", "substance", "object"]
    },
    "sharp": {
        "intelligent": ["mind", "edge",'student', "knife", "object"],
        "pointy": ["knife", "spike", "object", "pen"]
    },
    "warm": {
        "temperature": ["room", "blanket", "fire", "welcome"],
        "friendly": ["greeting", "host", "behavior", "attitude"]
    },
    "heavy": {
        "weight": ["box", "load", "burden", "work"],
        "intense": ["work", "emotion", "workload","storm", "heat"]
    },
    "soft": {
        "texture": ["pillow", "fabric", "voice", "touch"],
        "gentle": ["touch", "voice", "personality", "care"]
    },
    "sweet": {
        "taste": ["cake", "taste", "person","dessert", "gesture"],
        "kind": ["gesture", "person", "compliment", "behavior"]
    },
    "cool": {
        "temperature": ["breeze", "style", "temperature", "shirt"],
        "fashionable": ["outfit", "accessory", "trend", "design"]
    },
    "fast": {
        "speed": ["car", "response", "runner", "machine"],
        "quickly": ["reply", "task", "action", "delivery"]
    },
    "flat": {
        "surface": ["table", "surface", "land", "design"],
        "monotonous": ["lecture", "task", "color", "day"]
    },
    "deep": {
        "profound": ["thought", "well","thoughts", "conversation"],
        "distance": [ "sea", "distance", "size"]
    },
    "clear": {
        "transparent": ["water", "answer","glass", "sky"],
        "understandable": ["clear", "simple","instructions", "direct"]
    },
    "loud": {
        "volume": ["music", "noise", "voice", "sound"],
        "bold": ["statement", "move", "color", "person"]
    },
    "dark": {
        "lack of light": ["room", "mood", "color", "weather"],
        "gloomy": ["day", "outlook", "weather", "atmosphere"]
    },
    "clean": {
        "hygienic": ["room", "surface", "clothes", "hands"],
        "honest": ["person","record", "statement", "work", "behavior"]
    },
    "high": {
        "tall": ["mountain", "building", "score"],
        "extreme": ["temperature", "top", "max"]
    },
    "rich": {
        "wealthy": ["businessman", "sauce", "man", "flavor"],
        "flavorful": ["sauce", "food", "taste", "dish"]
    },
    "weak": {
        "physically feeble": ["individual", "health", "muscle", "condition"],
        "flimsy": ["child", "argument", "person", "structure"]
    },
    "dry": {
        "arid": ["desert", "climate", "response", "humor"],
        "lacking emotion": ["response", "tone", "style", "communication"]
    }
}

# Example Input Synonym Map
synonym_map = {
    "bright": {
        "intelligent": ["genius", "clever", "smart"],
        "shining": ["luminous", "glowing", "radiant"]
    },
    "cold": {
        "unfriendly": ["aloof", "distant", "detached"],
        "temperature": ["chilly", "freezing", "icy"]
    },
    "light": {
        "illumination": ["brightness", "radiance", "glow"],
        "weight": ["weightless", "featherweight", "lightweight"]
    },
    "hard": {
        "difficult": ["challenging", "arduous", "tough"],
        "solid": ["sturdy", "rigid", "firm"]
    },
    "sharp": {
        "intelligent": ["astute", "shrewd", "keen"],
        "pointy": ["acute", "spiked", "razor-sharp"]
    },
    "warm": {
        "temperature": ["hot", "toasty", "heated"],
        "friendly": ["cordial", "hospitable", "welcoming"]
    },
    "heavy": {
        "weight": ["weighty", ["massive", "burdensome", "ponderous"]],
        "intense": ["severe", "extreme", "strong"]
    },
    "soft": {
        "texture": ["fluffy", "pillowy", "smooth"],
        "gentle": ["tender", "kind", "mild"]
    },
    "sweet": {
        "taste": ["sugary", "honeyed", "saccharine"],
        "kind": ["benevolent", "gracious", "thoughtful"]
    },
    "cool": {
        "temperature": ["chilly", "refreshing", "crisp"],
        "fashionable": ["stylish", "trendy", "hip"]
    },
    "fast": {
        "speed": ["swift", "quick", "rapid"],
        "quickly": ["promptly", "immediately", "instantly"]
    },
    "flat": {
        "surface": ["even", "level", "plane"],
        "monotonous": ["dull", "tedious", "unvaried"]
    },
    "deep": {
        "profound": ["thoughtful", "meaningful", "substantial"],
        "distance": ["far", "distant", "remote"]
    },
    "clear": {
        "transparent": ["obvious", "evident", "apparent"],
        "understandable": ["comprehensible", "explicit", "lucid"]
    },
    "loud": {
        "volume": ["noisy", "boisterous", "clamorous"],
        "bold": ["strident", "assertive", "emphatic"]
    },
    "dark": {
        "lack of light": ["dim", "shadowy", "gloomy"],
        "gloomy": ["depressing", "morose", "dismal"]
    },
    "clean": {
        "hygienic": ["sanitary", "neat", "tidy"],
        "honest": ["upright", "truthful", "genuine"]
    },
    "high": {
        "tall": ["elevated", "towering", "lofty"],
        "extreme": ["maximum", "utmost", "supreme"]
    },
    "rich": {
        "wealthy": ["affluent", "opulent", "prosperous"],
        "flavorful": ["tasty", "savory", "delicious"]
    },
    "weak": {
        "physically feeble": ["fragile", "delicate", "infirm"],
        "flimsy": ["insecure", "unsteady", "unstable"]
    },
    "dry": {
        "arid": ["parched", "desiccated", "dehydrated"],
        "lacking emotion": ["stoic", "bland", "insipid"]
    }
}

# Tokenization
tokens = tokenize(sentences)
# Create vocabulary
vocabulary = create_vocabulary(tokens)
# Initialize vectors
root_words = ["bright", "cold", "light", "hard", "sharp", "warm", "heavy", "soft", "sweet", "cool", "fast", "flat", "deep", "clear", "loud", "dark", "clean", "high", "rich", "weak", "dry"]
vectors = initialize_vectors(vocabulary, root_words)
# Create clusters
clusters = create_clusters(vocabulary, context_map, synonym_map, vectors, root_words)


# Continuously process input sentences
while True:
    # Get input from the user
    input_sentence = input("Enter a sentence (or type 'exit' to quit): ")
    
    # Exit condition
    if input_sentence.lower() == 'exit':
        print("Exiting the program.")
        break

    # Process the input sentence to retrieve subclusters
    subcluster = process_input_subcluster(input_sentence, clusters, vectors)

    print("Subcluster for the input sentence:")
    print(subcluster)

Root word identified: bright
Adjacent words selected: ['sun', 'bright']
Subcluster for the input sentence:
{'shining': {'related_words': ['sun', 'light', 'lamp'], 'synonyms': ['luminous', 'glowing', 'radiant']}}
Exiting the program.
